# 🔗 Workflow Integration — Complete SAR Processing System

This notebook runs the **end-to-end SAR processing workflow**: both AI agents together with human-in-the-loop review and full audit trails.

## 📋 Business Context
This workflow simulates how banks actually process suspicious activity reports:
1. **Risk Screening**: AI agents analyze transaction patterns for suspicious activity
2. **Human Review**: Compliance officers review AI findings before proceeding
3. **Narrative Generation**: Only approved cases get full compliance documentation
4. **SAR Filing**: Complete regulatory forms are generated for submission
5. **Audit Documentation**: Every decision is logged for regulatory examination

## 🏗️ System Architecture

```
📊 CSV Data → 🔍 Risk Analyst → 👤 Human Decision → ✅ Compliance Officer → 📄 SAR Document
              (Chain-of-Thought)    (Gate)         (ReACT Framework)     (FinCEN Ready)
```

## 🚀 Prerequisites

Ensure these components are in place:
- ✅ Foundation module (`foundation_sar.py`) — schemas, data loader, audit logging
- ✅ Risk Analyst Agent (`risk_analyst_agent.py`)
- ✅ Compliance Officer Agent (`compliance_officer_agent.py`)
- ✅ Tests passing: `python -m pytest tests/ -v`

If anything is missing, implement the modules or run the earlier notebooks first.

In [1]:
# Setup and Environment Configuration
import os
import sys
import json
import pandas as pd
import uuid
import hashlib
from datetime import datetime, timedelta
from dotenv import load_dotenv

# Add src directory to Python path for module imports
sys.path.append(os.path.abspath('../src'))

# Load environment variables
load_dotenv('../env.txt')

print("📚 Libraries imported successfully!")
print("🔐 Environment variables loaded")
print("📂 Source directory added to Python path")

📚 Libraries imported successfully!
🔐 Environment variables loaded
📂 Source directory added to Python path


In [2]:
# OpenAI client setup
import openai

openai_api_key = os.getenv('OPENAI_API_KEY')
base_url = os.getenv('OPENAI_BASE_URL')  # Optional: for custom endpoint (e.g. proxy)

if not openai_api_key:
    print("⚠️ WARNING: No OpenAI API key found!")
    print("Please set OPENAI_API_KEY in your .env file")
else:
    client = openai.OpenAI(api_key=openai_api_key, base_url=base_url if base_url else None)
    print("✅ OpenAI client initialized")
    print(f"🔑 API key: {openai_api_key[:8]}...{openai_api_key[-4:]}")
    if base_url:
        print(f"📍 Base URL: {base_url}")

✅ OpenAI client initialized with Vocareum routing
🔑 API key: voc-2045...0158
📍 Base URL: https://openai.vocareum.com/v1


In [3]:
# Import SAR components
# Foundation components and AI agents for the SAR workflow

from foundation_sar import (
    CustomerData,
    AccountData,
    TransactionData,
    CaseData,
    RiskAnalystOutput,
    ComplianceOfficerOutput,
    ExplainabilityLogger,
    DataLoader,
    load_csv_data
)
from risk_analyst_agent import RiskAnalystAgent
from compliance_officer_agent import ComplianceOfficerAgent

# Create agent instances
os.makedirs("../outputs/audit_logs", exist_ok=True)
explainability_logger = ExplainabilityLogger("../outputs/audit_logs/workflow_integration.jsonl")
risk_agent = RiskAnalystAgent(client, explainability_logger)
compliance_agent = ComplianceOfficerAgent(client, explainability_logger)

print("✅ Components imported successfully")

✅ Components imported successfully


## 📊 Step 1: Data Loading and Preprocessing

Load the financial data and prepare it for analysis.

In [4]:
# Load and Preprocess Financial Data
# Load customer, account, and transaction data from CSV

def load_and_preprocess_data():
    """
    Load CSV data and prepare for analysis.
    
    1. Load customers.csv, accounts.csv, transactions.csv from ../data/
    2. Handle missing values in optional fields
    3. Create data dictionaries for processing
    4. Return cleaned datasets
    """
    print("📊 Loading Financial Data")
    print("✅ Load CSV files from ../data/ directory")
    print("✅ Handle NaN values in optional fields")
    print("✅ Convert to dictionaries for processing")
    
    # Load CSVs and normalize optional fields
    customers_df = pd.read_csv("../data/customers.csv", dtype={'ssn_last_4': str})
    accounts_df = pd.read_csv("../data/accounts.csv")
    transactions_df = pd.read_csv("../data/transactions.csv")
    
    # Handle NaN values
    transactions_df['counterparty'] = transactions_df['counterparty'].fillna('')
    transactions_df['location'] = transactions_df['location'].fillna('')
    customers_df['phone'] = customers_df['phone'].fillna('')
    
    # Convert to dictionaries
    customers_data = customers_df.to_dict('records')
    accounts_data = accounts_df.to_dict('records')
    transactions_data = transactions_df.to_dict('records')
    
    print(f"📈 Loaded: {len(customers_data)} customers, {len(accounts_data)} accounts, {len(transactions_data)} transactions")
    return customers_data, accounts_data, transactions_data

# Load data
customers_data, accounts_data, transactions_data = load_and_preprocess_data()

📊 Loading Financial Data
✅ Load CSV files from ../data/ directory
✅ Handle NaN values in optional fields
✅ Convert to dictionaries for processing
📈 Loaded: 150 customers, 178 accounts, 4268 transactions


## 🎯 Step 2: Customer Risk Screening

Implement intelligent customer screening to identify high-risk cases for detailed analysis.

In [5]:
def screen_high_risk_customers(customers_data, accounts_data, transactions_data, top_n=5):
    """
    Risk-based customer screening.
    
    Screening criteria:
    1. High risk ratings (Medium, High)
    2. Large transaction amounts (>$100K total)
    3. High transaction frequency (>50 transactions)
    4. Recent activity patterns
    
    Returns top N highest-risk customers for detailed analysis.
    """
    print("🔍 Customer Risk Screening")
    
    selected_customers = []

    for customer in customers_data:
        # Get customer accounts and transactions
        customer_accounts = [acc for acc in accounts_data if acc['customer_id'] == customer['customer_id']]
        customer_transactions = [txn for txn in transactions_data if any(txn['account_id'] == acc['account_id'] for acc in customer_accounts)]

        # Calculate risk indicators
        total_amount = sum(abs(txn['amount']) for txn in customer_transactions)
        transaction_count = len(customer_transactions)
        risk_rating = customer['risk_rating']

        # Apply screening criteria
        risk_flags = []
        if risk_rating in ['Medium', 'High']:
            risk_flags.append('high_risk_rating')
        if total_amount > 100000:
            risk_flags.append('large_amounts')
        if transaction_count > 50:
            risk_flags.append('high_frequency')

        # Select high-risk customers
        if len(risk_flags) >= 2:  # Multiple risk flags
            selected_customers.append({
                'customer': customer,
                'accounts': customer_accounts,
                'transactions': customer_transactions,
                'total_amount': total_amount,
                'transaction_count': transaction_count,
                'risk_flags': risk_flags
            })

    # Sort by risk score and take top N
    selected_customers.sort(key=lambda x: (len(x['risk_flags']), x['total_amount']), reverse=True)
    top_customers = selected_customers[:top_n]

    print(f"📊 Screened {len(customers_data)} customers")
    print(f"📊 Flagged  {len(selected_customers)} with 2+ risk flags")
    print(f"📊 Selected {len(top_customers)} for SAR analysis")
    for i, c in enumerate(top_customers, 1):
        print(f"   {i}. {c['customer']['name']} | Flags: {c['risk_flags']} | Volume: ${c['total_amount']:,.2f}")

    return top_customers


# Run customer screening
selected_customers = screen_high_risk_customers(customers_data, accounts_data, transactions_data)

🔍 Customer Risk Screening
📊 Screened 150 customers
📊 Flagged  26 with 2+ risk flags
📊 Selected 5 for SAR analysis
   1. Jacqueline Rodriguez | Flags: ['high_risk_rating', 'large_amounts', 'high_frequency'] | Volume: $386,834.91
   2. Michael Stanley | Flags: ['high_risk_rating', 'large_amounts', 'high_frequency'] | Volume: $380,633.78
   3. Cindy Clayton | Flags: ['high_risk_rating', 'large_amounts', 'high_frequency'] | Volume: $286,342.62
   4. Tracy Lewis | Flags: ['high_risk_rating', 'large_amounts', 'high_frequency'] | Volume: $274,025.18
   5. Patrick Williams | Flags: ['high_risk_rating', 'large_amounts', 'high_frequency'] | Volume: $222,070.26


## 🤖 Step 3: Two-Stage AI Analysis with Human Gates

Implement the core two-stage workflow:
1. **Stage 1**: Risk Analyst performs Chain-of-Thought analysis
2. **Human Gate**: Review and decision to proceed
3. **Stage 2**: Compliance Officer generates ReACT narratives (only if approved)

In [6]:
# SAR document helpers — must be defined before running the workflow cell below
def create_sar_document(case_data, risk_analysis, compliance_review):
    """Create complete SAR document for regulatory submission."""
    sar_id = f"SAR_{uuid.uuid4()}"
    filing_date = datetime.now().isoformat()
    sar_document = {
        'sar_id': sar_id,
        'sar_metadata': {
            'sar_id': sar_id,
            'filing_date': filing_date,
            'filing_type': 'Suspicious Activity Report',
            'ai_generated': True,
            'review_status': 'human_approved'
        },
        'subject_information': {
            'customer_name': case_data.customer.name,
            'customer_id': case_data.customer.customer_id,
            'address': case_data.customer.address,
            'customer_since': case_data.customer.customer_since,
            'risk_rating': case_data.customer.risk_rating
        },
        'suspicious_activity': {
            'classification': risk_analysis.classification,
            'risk_level': risk_analysis.risk_level,
            'confidence_score': risk_analysis.confidence_score,
            'narrative': compliance_review.narrative,
            'key_indicators': risk_analysis.key_indicators,
            'ai_reasoning': risk_analysis.reasoning
        },
        'regulatory_compliance': {
            'citations': getattr(compliance_review, 'regulatory_citations', []),
            'narrative_word_count': len(compliance_review.narrative.split()),
            'compliance_status': 'approved'
        },
        'audit_trail': {
            'case_id': case_data.case_id,
            'processing_date': filing_date,
            'ai_agents_used': ['RiskAnalyst', 'ComplianceOfficer'],
            'human_reviewer': 'compliance_officer'
        }
    }
    return sar_document

def save_sar_document(sar_document):
    """Save SAR document to outputs directory."""
    os.makedirs("../outputs/filed_sars", exist_ok=True)
    filename = f"../outputs/filed_sars/{sar_document['sar_metadata']['sar_id']}.json"
    with open(filename, 'w') as f:
        json.dump(sar_document, f, indent=2)

print("✅ create_sar_document and save_sar_document defined")

✅ create_sar_document and save_sar_document defined


In [7]:
# Two-Stage AI Workflow
# Risk Analyst → Human Gate → Compliance Officer

def run_two_stage_sar_workflow(selected_customers):
    """
    Complete two-stage SAR processing workflow with human decision gates.
    
    For each customer:
    1. Create CaseData from customer, accounts, transactions
    2. Run Risk Analyst analysis (Chain-of-Thought)
    3. Present findings to human reviewer
    4. Get human decision (proceed/reject)
    5. If approved: Run Compliance Officer (ReACT) and generate SAR
    6. Save SAR to ../outputs/filed_sars/ and log decision for audit
    """
    print("🤖 Two-Stage SAR Processing Workflow")
    print("✅ Risk Analyst → Human Gate → Compliance Officer (implemented)")
    
    # Initialize tracking
    processed_cases = []
    approved_sars = []
    rejected_cases = []
    audit_decisions = []
    
    # Process each selected customer
    for i, customer_data in enumerate(selected_customers, 1):
        print(f"\n🔍 CUSTOMER {i}/{len(selected_customers)}: {customer_data['customer']['name']}")
        print("=" * 60)
        
        try:
            # Create case data
            loader = DataLoader(explainability_logger)
            case_data = loader.create_case_from_data(
                customer_data['customer'],
                customer_data['accounts'], 
                customer_data['transactions']
            )
            
            # STAGE 1: Risk Analysis
            print("🔍 STAGE 1: Risk Analysis")
            risk_analysis = risk_agent.analyze_case(case_data)
            
            # Display analysis results
            print(f"Classification: {risk_analysis.classification}")
            print(f"Confidence: {risk_analysis.confidence_score:.2f}")
            print(f"Risk Level: {risk_analysis.risk_level}")
            print(f"Reasoning: {risk_analysis.reasoning}")
            
            # HUMAN DECISION GATE
            decision = input("🤔 Proceed with SAR filing? (yes/no): ").strip().lower()
            should_proceed = decision in ['yes', 'y']
            
            if should_proceed:
                # STAGE 2: Compliance Narrative
                print("📝 STAGE 2: Compliance Narrative Generation")
                compliance_review = compliance_agent.generate_compliance_narrative(case_data, risk_analysis)
                
                # Generate complete SAR document
                sar_document = create_sar_document(case_data, risk_analysis, compliance_review)
                
                # Save SAR
                save_sar_document(sar_document)
                approved_sars.append(sar_document)
                
                print(f"✅ SAR FILED: {sar_document['sar_id']}")
            else:
                rejected_cases.append({'case_id': case_data.case_id, 'reason': 'human_rejection'})
                print("❌ SAR REJECTED by human reviewer")
            
            # Log decision and record processed case
            audit_decisions.append({
                'case_id': case_data.case_id,
                'customer_name': case_data.customer.name,
                'decision': 'PROCEED' if should_proceed else 'REJECT',
                'ai_classification': risk_analysis.classification,
                'ai_confidence': risk_analysis.confidence_score,
                'reviewer_decision': decision
            })
            processed_cases.append({
                'case_id': case_data.case_id,
                'customer_name': case_data.customer.name,
                'approved': should_proceed
            })
            
        except Exception as e:
            print(f"❌ Error processing customer: {e}")
    
    return processed_cases, approved_sars, rejected_cases, audit_decisions

# Run the complete workflow
processed_cases, approved_sars, rejected_cases, audit_decisions = run_two_stage_sar_workflow(selected_customers)

🤖 Two-Stage SAR Processing Workflow
✅ Risk Analyst → Human Gate → Compliance Officer (implemented)

🔍 CUSTOMER 1/5: Jacqueline Rodriguez
🔍 STAGE 1: Risk Analysis


Classification: Money_Laundering
Confidence: 0.90
Risk Level: High
Reasoning: STEP 1: High-risk customer with significant income and active accounts. STEP 2: Large outgoing wire transfers totaling over $49,000 in a short period, inconsistent with occupation. STEP 3: BSA/AML regulations require SAR for potential money laundering. STEP 4: High confidence due to volume and frequency of suspicious transactions. STEP 5: Classification as Money_Laundering due to layering of funds.
📝 STAGE 2: Compliance Narrative Generation
✅ SAR FILED: SAR_57918133-d822-4e55-81b3-7b8f40299bb0

🔍 CUSTOMER 2/5: Michael Stanley
🔍 STAGE 1: Risk Analysis
Classification: Money_Laundering
Confidence: 0.85
Risk Level: High
Reasoning: STEP 1: Customer is an investment banker with medium risk, high account balances, and high transaction volumes. STEP 2: Unusual high-value transactions, rapid movement of funds, and large outgoing wire transfers. STEP 3: BSA/AML regulations suggest SAR filing due to suspicious activity.

## 📄 Step 4: SAR Document Generation

Create complete, FinCEN-ready SAR documents with all required metadata.

In [8]:
# SAR Document Generation — FinCEN-ready documents with required metadata

def create_sar_document(case_data, risk_analysis, compliance_review):
    """
    Create complete SAR document: metadata, subject, activity, narrative,
    regulatory citations, and audit trail.
    """
    sar_id = f"SAR_{uuid.uuid4()}"
    filing_date = datetime.now().isoformat()
    
    sar_document = {
        'sar_id': sar_id,
        'sar_metadata': {
            'sar_id': sar_id,
            'filing_date': filing_date,
            'filing_type': 'Suspicious Activity Report',
            'ai_generated': True,
            'review_status': 'human_approved'
        },
        'subject_information': {
            'customer_name': case_data.customer.name,
            'customer_id': case_data.customer.customer_id,
            'address': case_data.customer.address,
            'customer_since': case_data.customer.customer_since,
            'risk_rating': case_data.customer.risk_rating
        },
        'suspicious_activity': {
            'classification': risk_analysis.classification,
            'risk_level': risk_analysis.risk_level,
            'confidence_score': risk_analysis.confidence_score,
            'narrative': compliance_review.narrative,
            'key_indicators': risk_analysis.key_indicators,
            'ai_reasoning': risk_analysis.reasoning
        },
        'regulatory_compliance': {
            'citations': getattr(compliance_review, 'regulatory_citations', []),
            'narrative_word_count': len(compliance_review.narrative.split()),
            'compliance_status': 'approved'
        },
        'audit_trail': {
            'case_id': case_data.case_id,
            'processing_date': filing_date,
            'ai_agents_used': ['RiskAnalyst', 'ComplianceOfficer'],
            'human_reviewer': 'compliance_officer'
        }
    }
    
    return sar_document

def save_sar_document(sar_document):
    """Save SAR document to ../outputs/filed_sars/ directory."""
    os.makedirs("../outputs/filed_sars", exist_ok=True)
    filename = f"../outputs/filed_sars/{sar_document['sar_metadata']['sar_id']}.json"
    with open(filename, 'w') as f:
        json.dump(sar_document, f, indent=2)
    sid = sar_document.get('sar_id') or sar_document['sar_metadata']['sar_id']
    print(f"✅ SAR saved: {sid}.json")

print("✅ SAR document generation functions defined")

✅ SAR document generation functions defined


## 📊 Step 5: Workflow Metrics and Analysis

Analyze the efficiency and effectiveness of the SAR processing workflow.

In [9]:
# Workflow Analysis and Metrics
# Efficiency metrics, cost optimization, and AI decision validation

def analyze_workflow_efficiency(processed_cases, approved_sars, rejected_cases, audit_decisions):
    """
    Calculate workflow efficiency: processing counts, approval/rejection rates,
    and cost optimization from two-stage processing.
    """
    print("📊 Workflow Efficiency Analysis")
    print("✅ Processing metrics calculated")
    
    total_cases = len(processed_cases)
    approved_cases = len(approved_sars)
    rejected_cases_count = len(rejected_cases)
    
    if total_cases > 0:
        approval_rate = approved_cases / total_cases
        rejection_rate = rejected_cases_count / total_cases
    else:
        approval_rate = rejection_rate = 0
    
    print(f"📈 WORKFLOW METRICS:")
    print(f"   Total Cases Processed: {total_cases}")
    print(f"   SARs Filed: {approved_cases}")
    print(f"   Cases Rejected: {rejected_cases_count}")
    print(f"   Approval Rate: {approval_rate:.1%}")
    print(f"   Rejection Rate: {rejection_rate:.1%}")
    
    # Cost optimization analysis
    print(f"\n💰 COST OPTIMIZATION:")
    print(f"   Two-stage processing saves costs by only running")
    print(f"   expensive compliance generation on approved cases")
    print(f"   Cost savings: {rejection_rate:.1%} of compliance calls avoided")

def validate_ai_decisions(audit_decisions):
    """Analyze AI decision patterns: classifications, confidence scores, human overrides."""
    if not audit_decisions:
        print("📋 No audit decisions to analyze (run workflow first)")
        return
    print("\n📊 AI Decision Analysis")
    print("✅ Classification accuracy and confidence distributions")
    # Classification distribution
    from collections import Counter
    classifications = Counter(d.get('ai_classification') for d in audit_decisions)
    print("   Classification distribution:")
    for cls, count in classifications.most_common():
        print(f"     • {cls}: {count}")
    # Confidence stats
    confidences = [d.get('ai_confidence') for d in audit_decisions if d.get('ai_confidence') is not None]
    if confidences:
        avg_conf = sum(confidences) / len(confidences)
        print(f"   Confidence: avg={avg_conf:.2f}, min={min(confidences):.2f}, max={max(confidences):.2f}")
    # Human override pattern (rejected when AI flagged)
    rejected = sum(1 for d in audit_decisions if d.get('decision') == 'REJECT')
    print(f"   Human rejections (override): {rejected} of {len(audit_decisions)} cases")

# Run analysis
analyze_workflow_efficiency(processed_cases, approved_sars, rejected_cases, audit_decisions)
validate_ai_decisions(audit_decisions)

📊 Workflow Efficiency Analysis
✅ Processing metrics calculated
📈 WORKFLOW METRICS:
   Total Cases Processed: 5
   SARs Filed: 5
   Cases Rejected: 0
   Approval Rate: 100.0%
   Rejection Rate: 0.0%

💰 COST OPTIMIZATION:
   Two-stage processing saves costs by only running
   expensive compliance generation on approved cases
   Cost savings: 0.0% of compliance calls avoided

📊 AI Decision Analysis
✅ Classification accuracy and confidence distributions
   Classification distribution:
     • Structuring: 3
     • Money_Laundering: 2
   Confidence: avg=0.89, min=0.85, max=0.95
   Human rejections (override): 0 of 5 cases


## 🏁 Step 6: Complete System Demonstration

Run the complete workflow with multiple scenarios to validate end-to-end behavior.

In [10]:
# Complete System Demonstration
# Run full workflow, generate SARs and audit reports, show efficiency metrics

def demonstrate_complete_system():
    """
    Run complete system: load data, screen customers, two-stage workflow
    with human gates, generate SARs, audit trail, and efficiency metrics.
    """
    print("🏁 Complete SAR Processing System Demonstration")
    print("✅ Run complete workflow with multiple customers")
    print("✅ Approval and rejection scenarios via human gate")
    print("✅ Audit reports and efficiency metrics")
    
    print("🚀 Running complete system test...")
    
    # Load fresh data
    customers_data, accounts_data, transactions_data = load_and_preprocess_data()
    
    # Screen customers
    selected_customers = screen_high_risk_customers(customers_data, accounts_data, transactions_data, top_n=3)
    
    # Run workflow
    processed_cases, approved_sars, rejected_cases, audit_decisions = run_two_stage_sar_workflow(selected_customers)
    
    # Generate final report
    analyze_workflow_efficiency(processed_cases, approved_sars, rejected_cases, audit_decisions)
    
    print(f"🎉 System demonstration complete!")
    print(f"📄 SAR documents saved to: ../outputs/filed_sars/")
    print(f"📊 Audit logs saved to: ../outputs/audit_logs/")

demonstrate_complete_system()

🏁 Complete SAR Processing System Demonstration
✅ Run complete workflow with multiple customers
✅ Approval and rejection scenarios via human gate
✅ Audit reports and efficiency metrics
🚀 Running complete system test...
📊 Loading Financial Data
✅ Load CSV files from ../data/ directory
✅ Handle NaN values in optional fields
✅ Convert to dictionaries for processing
📈 Loaded: 150 customers, 178 accounts, 4268 transactions
🔍 Customer Risk Screening
📊 Screened 150 customers
📊 Flagged  26 with 2+ risk flags
📊 Selected 3 for SAR analysis
   1. Jacqueline Rodriguez | Flags: ['high_risk_rating', 'large_amounts', 'high_frequency'] | Volume: $386,834.91
   2. Michael Stanley | Flags: ['high_risk_rating', 'large_amounts', 'high_frequency'] | Volume: $380,633.78
   3. Cindy Clayton | Flags: ['high_risk_rating', 'large_amounts', 'high_frequency'] | Volume: $286,342.62
🤖 Two-Stage SAR Processing Workflow
✅ Risk Analyst → Human Gate → Compliance Officer (implemented)

🔍 CUSTOMER 1/3: Jacqueline Rodrigue

## 📝 Workflow Checklist

### ✅ Components Covered
- [ ] **Data Loading**: Load and preprocess CSV data with proper error handling
- [ ] **Customer Screening**: Implement risk-based screening to identify high-risk cases
- [ ] **Two-Stage Workflow**: Build complete Risk Analyst → Human Gate → Compliance Officer flow
- [ ] **Human Decision Gates**: Implement interactive approval/rejection points
- [ ] **SAR Document Generation**: Create complete FinCEN-ready documents with metadata
- [ ] **Audit Trail Creation**: Log all decisions and reasoning for regulatory examination
- [ ] **Efficiency Metrics**: Calculate cost optimization and processing efficiency
- [ ] **System Demonstration**: Test complete workflow with multiple scenarios

### ✅ Testing and Validation Requirements
- [ ] **Component Validation**: Verify all foundation components and agents are available
- [ ] **Integration Testing**: Run comprehensive test suites for all components with proper sys.path setup
- [ ] **End-to-End Testing**: Test complete workflow with automated scenarios
- [ ] **Error Handling Testing**: Validate graceful handling of edge cases and failures
- [ ] **Output Validation**: Ensure SAR documents meet regulatory standards
- [ ] **Performance Testing**: Measure workflow efficiency and processing times

### ✅ Technical Requirements
- [ ] **Error Handling**: Robust exception handling for all workflow steps
- [ ] **Data Validation**: Proper validation of all inputs and outputs
- [ ] **File Management**: Organize outputs in appropriate directories
- [ ] **Logging**: Comprehensive audit logging for compliance
- [ ] **Performance**: Efficient processing of multiple cases
- [ ] **User Experience**: Clear prompts and feedback for human reviewers
- [ ] **Test Infrastructure**: Proper test imports and sys.path configuration

### ✅ Business Requirements  
- [ ] **Regulatory Compliance**: Ensure all SAR documents meet FinCEN requirements
- [ ] **Cost Optimization**: Demonstrate savings from two-stage processing
- [ ] **Audit Readiness**: Create examination-ready documentation
- [ ] **Quality Assurance**: Validate AI decisions with human oversight
- [ ] **Scalability**: Design for processing larger datasets
- [ ] **Production Readiness**: Complete testing validates system reliability

## 🎯 What the Integrated System Provides

When complete, the workflow:
- ✅ Process real financial data with proper validation
- ✅ Execute complete two-stage AI workflow with human gates
- ✅ Generate regulatory-compliant SAR documents
- ✅ Create comprehensive audit trails for all decisions
- ✅ Demonstrate measurable cost optimization benefits
- ✅ Handle errors gracefully and provide clear user feedback
- ✅ Pass all integration and end-to-end tests
- ✅ Meet production-ready quality standards

## 🚀 Next Steps

1. **Complete Implementation**: Fill in all TODO sections with working code
2. **Run Integration Tests**: Validate all components work together properly
3. **Execute End-to-End Tests**: Test complete workflow with automated scenarios
4. **Test Thoroughly**: Run complete workflow with various manual scenarios
5. **Validate Outputs**: Ensure SAR documents meet regulatory requirements
6. **Document Results**: Create final project documentation and metrics
7. **Optional**: Document or present the system's capabilities and business value

This completes the SAR workflow integration.

## 🧪 Step 7: Workflow Testing and Validation

Validate the workflow using the test suite and integration checks below.

In [11]:
# 🧪 Workflow Integration Testing
# Validate the workflow with integration tests

import sys
import os

# Add project root, src, and tests so agents and test modules resolve the same code
project_root = os.path.abspath('..')
src_path = os.path.join(project_root, 'src')
tests_path = os.path.join(project_root, 'tests')
for p in (project_root, src_path, tests_path):
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"📁 Added tests directory to Python path: {tests_path}")
print(f"📁 Project root on path: {project_root}")


def run_integration_tests():
    """
    Run comprehensive integration tests to validate the complete workflow
    
    Tests include:
    1. Foundation components integration
    2. Agent communication and data flow
    3. End-to-end workflow execution
    4. Error handling and edge cases
    5. Output validation and compliance
    """
    print("🧪 Comprehensive Integration Testing")
    print("✅ Workflow implemented — uncomment pytest block below to run test suite")
    
    #Uncomment to run full test suite (requires pytest and test modules):
    try:
        # Import all test modules
        from test_foundation import TestCustomerData, TestAccountData, TestTransactionData, TestCaseData
        from test_risk_analyst import TestRiskAnalystAgent
        from test_compliance_officer import TestComplianceOfficerAgent
        import pytest
        
        print("🔍 Running Foundation Component Tests...")
        _cwd = os.getcwd()
        try:
            os.chdir(project_root)
            foundation_result = pytest.main([
                "tests/test_foundation.py", 
            "-v", 
                "--tb=short"
            ])
        finally:
            os.chdir(_cwd)
        
        print("🔍 Running Risk Analyst Agent Tests...")
        _cwd = os.getcwd()
        try:
            os.chdir(project_root)
            risk_result = pytest.main([
                "tests/test_risk_analyst.py", 
                "-v", 
                "--tb=short"
            ])
        finally:
            os.chdir(_cwd)
        
        print("📝 Running Compliance Officer Agent Tests...")
        _cwd = os.getcwd()
        try:
            os.chdir(project_root)
            compliance_result = pytest.main([
                "tests/test_compliance_officer.py", 
                "-v", 
                "--tb=short"
            ])
        finally:
            os.chdir(_cwd)
        
        # Calculate overall test results
        all_passed = foundation_result == 0 and risk_result == 0 and compliance_result == 0
        
        print("\n" + "="*60)
        print("📊 INTEGRATION TEST RESULTS:")
        print(f"   Foundation Components: {'✅ PASS' if foundation_result == 0 else '❌ FAIL'}")
        print(f"   Risk Analyst Agent: {'✅ PASS' if risk_result == 0 else '❌ FAIL'}")
        print(f"   Compliance Officer Agent: {'✅ PASS' if compliance_result == 0 else '❌ FAIL'}")
        print(f"   Overall Status: {'✅ ALL TESTS PASSED' if all_passed else '❌ SOME TESTS FAILED'}")
        
        if all_passed:
            print("\n🎉 System is ready for production workflow testing!")
            print("📝 Proceed to run the complete system demonstration.")
        else:
            print("\n⚠️ Fix failing tests before running the complete workflow.")
            print("📝 Return to previous notebooks to fix component issues.")
        
        return all_passed
            
    except ImportError as e:
        print(f"❌ Import Error: {e}")
        print("💡 Make sure all components are implemented:")
        print("   • foundation_sar.py")
        print("   • risk_analyst_agent.py") 
        print("   • compliance_officer_agent.py")
        return False

def validate_workflow_components():
    """Validate that all required components are available for integration"""
    print("🔍 Validating Workflow Components")
    
    components_status = {
        'foundation_sar': False,
        'risk_analyst_agent': False,
        'compliance_officer_agent': False,
        'test_modules': False
    }
    
    try:
        # Check foundation components
        from foundation_sar import CustomerData, CaseData, ExplainabilityLogger, DataLoader
        components_status['foundation_sar'] = True
        print("✅ Foundation components available")
    except ImportError:
        print("❌ Foundation components not available")
    
    try:
        # Check risk analyst agent
        from risk_analyst_agent import RiskAnalystAgent
        components_status['risk_analyst_agent'] = True
        print("✅ Risk Analyst Agent available")
    except ImportError:
        print("❌ Risk Analyst Agent not available")
    
    try:
        # Check compliance officer agent
        from compliance_officer_agent import ComplianceOfficerAgent
        components_status['compliance_officer_agent'] = True
        print("✅ Compliance Officer Agent available")
    except ImportError:
        print("❌ Compliance Officer Agent not available")
    
    try:
        # Check test modules
        from test_foundation import TestCustomerData
        from test_risk_analyst import TestRiskAnalystAgent  
        from test_compliance_officer import TestComplianceOfficerAgent
        components_status['test_modules'] = True
        print("✅ Test modules available")
    except ImportError:
        print("❌ Test modules not available")
    
    all_ready = all(components_status.values())
    
    print(f"\n📊 Component Status: {'✅ ALL READY' if all_ready else '⚠️ INCOMPLETE'}")
    if not all_ready:
        print("💡 Complete missing components before running integration tests")
    
    return all_ready

# Run component validation
components_ready = validate_workflow_components()

# Run integration tests if components are ready
if components_ready:
    print("\n🚀 All components ready - you can run integration tests!")
    run_integration_tests()
else:
    print("\n📋 Complete component implementation first, then run integration tests")

📁 Added tests directory to Python path: /Users/ak/Documents/aiforfin/project/starter/tests
📁 Project root on path: /Users/ak/Documents/aiforfin/project/starter
🔍 Validating Workflow Components
✅ Foundation components available
✅ Risk Analyst Agent available
✅ Compliance Officer Agent available
✅ Test modules available

📊 Component Status: ✅ ALL READY

🚀 All components ready - you can run integration tests!
🧪 Comprehensive Integration Testing
✅ Workflow implemented — uncomment pytest block below to run test suite
🔍 Running Foundation Component Tests...
============================= test session starts ==============================
platform darwin -- Python 3.13.9, pytest-8.4.2, pluggy-1.5.0 -- /opt/anaconda3/bin/python
cachedir: .pytest_cache
rootdir: /Users/ak/Documents/aiforfin/project/starter
plugins: anyio-4.10.0
collecting ... collected 10 items

tests/test_foundation.py::TestCustomerData::test_valid_customer_data PASSED [ 10%]
tests/test_foundation.py::TestCustomerData::test_cust

In [12]:
# 🎯 End-to-End Workflow Testing
# Test the complete workflow with known test scenarios

def test_complete_workflow():
    """
    Test the complete SAR processing workflow end-to-end
    
    This test should:
    1. Load test data (can use a subset of actual data)
    2. Run customer screening
    3. Execute two-stage AI analysis
    4. Simulate human decisions (automated for testing)
    5. Generate SAR documents
    6. Validate all outputs
    7. Check audit trail completeness
    """
    print("🎯 End-to-End Workflow Testing")
    print("✅ Workflow components ready — running automated E2E test")
    
    try:
        print("🚀 Starting end-to-end workflow test...")
        print("📊 Loading test data...")
        customers_data, accounts_data, transactions_data = load_and_preprocess_data()
        if not customers_data:
            print("⚠️ No test data available")
            return False
        print("🔍 Testing customer screening...")
        selected_customers = screen_high_risk_customers(
            customers_data, accounts_data, transactions_data, top_n=2
        )
        if not selected_customers:
            print("⚠️ No customers selected")
            return False
        print(f"✅ Selected {len(selected_customers)} customers for testing")
        print("🤖 Testing automated workflow (Risk Analyst + Compliance Officer + SAR)...")
        test_results = {'cases_processed': 0, 'sars_generated': 0, 'errors': []}
        for customer_data in selected_customers:
            try:
                loader = DataLoader(explainability_logger)
                case_data = loader.create_case_from_data(
                    customer_data['customer'], customer_data['accounts'], customer_data['transactions']
                )
                risk_analysis = risk_agent.analyze_case(case_data)
                assert hasattr(risk_analysis, 'classification') and hasattr(risk_analysis, 'confidence_score')
                compliance_review = compliance_agent.generate_compliance_narrative(case_data, risk_analysis)
                assert hasattr(compliance_review, 'narrative')
                sar_document = create_sar_document(case_data, risk_analysis, compliance_review)
                assert sar_document and (sar_document.get('sar_id') or sar_document.get('sar_metadata', {}).get('sar_id'))
                test_results['cases_processed'] += 1
                test_results['sars_generated'] += 1
                print(f"✅ Processed: {customer_data['customer']['name']}")
            except Exception as e:
                test_results['errors'].append(f"{customer_data['customer']['name']}: {e}")
                print(f"❌ Error: {customer_data['customer']['name']}: {e}")
        print("\n📊 END-TO-END TEST RESULTS:")
        print(f"   Cases Processed: {test_results['cases_processed']}")
        print(f"   SARs Generated: {test_results['sars_generated']}")
        print(f"   Errors: {len(test_results['errors'])}")
        if test_results['errors']:
            for err in test_results['errors']:
                print(f"     • {err}")
        success = len(test_results['errors']) == 0 and test_results['cases_processed'] > 0
        print("\n🎉 END-TO-END TEST PASSED!" if success else "\n⚠️ END-TO-END TEST HAD ISSUES")
        return success
    except Exception as e:
        print(f"❌ End-to-end test failed: {e}")
        return False

# Run end-to-end test
test_success = test_complete_workflow()

🎯 End-to-End Workflow Testing
✅ Workflow components ready — running automated E2E test
🚀 Starting end-to-end workflow test...
📊 Loading test data...
📊 Loading Financial Data
✅ Load CSV files from ../data/ directory
✅ Handle NaN values in optional fields
✅ Convert to dictionaries for processing
📈 Loaded: 150 customers, 178 accounts, 4268 transactions
🔍 Testing customer screening...
🔍 Customer Risk Screening
📊 Screened 150 customers
📊 Flagged  26 with 2+ risk flags
📊 Selected 2 for SAR analysis
   1. Jacqueline Rodriguez | Flags: ['high_risk_rating', 'large_amounts', 'high_frequency'] | Volume: $386,834.91
   2. Michael Stanley | Flags: ['high_risk_rating', 'large_amounts', 'high_frequency'] | Volume: $380,633.78
✅ Selected 2 customers for testing
🤖 Testing automated workflow (Risk Analyst + Compliance Officer + SAR)...
✅ Processed: Jacqueline Rodriguez
✅ Processed: Michael Stanley

📊 END-TO-END TEST RESULTS:
   Cases Processed: 2
   SARs Generated: 2
   Errors: 0

🎉 END-TO-END TEST PASS